In [ ]:
import shutil

# Create the directory /kaggle/working/animal_dataset if it doesn't exist
!mkdir -p /kaggle/working/animal_dataset

# Copy the entire dataset folder from input directory to working directory
# dirs_exist_ok=True allows overwriting if destination folder already exists
shutil.copytree(
    "/kaggle/input/dadasdas/Data/Kangaroo-Koala.v14-kangaroo-koala-v13.yolov8", 
    "/kaggle/working/animal_dataset", 
    dirs_exist_ok=True
)

# Define a new YAML configuration string for the dataset paths and classes
new_yaml = """
train: /kaggle/working/animal_dataset/train/images
val: /kaggle/working/animal_dataset/valid/images
test: /kaggle/working/animal_dataset/test/images

nc: 2
names: ['Koala', 'kangaroo']
"""

# Write the YAML configuration string into a file data.yaml in the working directory
with open("/kaggle/working/animal_dataset/data.yaml", "w") as f:
    f.write(new_yaml)


In [ ]:
!pip install ultralytics
!pip install -q albumentations opencv-python


In [ ]:
import os
import cv2
import shutil
import albumentations as A

# --- 1. Define paths ---
image_dir = "/kaggle/working/animal_dataset/train/images"      # Original training images folder
label_dir = "/kaggle/working/animal_dataset/train/labels"      # Original YOLO label files folder
aug_img_dir = "/kaggle/working/animal_dataset/train/images_aug" # Augmented images output folder
aug_lbl_dir = "/kaggle/working/animal_dataset/train/labels_aug" # Augmented labels output folder

# Create augmented folders if they do not exist
os.makedirs(aug_img_dir, exist_ok=True)
os.makedirs(aug_lbl_dir, exist_ok=True)

# --- 2. Define augmentation pipeline using Albumentations ---
transform = A.Compose([
    A.HorizontalFlip(p=0.5),                # Flip horizontally with 50% probability
    A.RandomBrightnessContrast(p=0.3),     # Random brightness/contrast changes with 30% probability
    A.Blur(blur_limit=3, p=0.2),           # Blur effect with limit 3, 20% probability
    A.RandomGamma(p=0.3)                   # Random gamma adjustment with 30% probability
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))
# bbox_params tells Albumentations that bounding boxes are in YOLO format (x_center, y_center, width, height normalized 0-1)

# --- 3. Function to validate bounding box coordinates ---
def is_valid_bbox(x, y, w, h):
    """
    Check if bounding box is valid (non-zero size and within image boundaries)
    All coordinates normalized between 0 and 1 in YOLO format.
    """
    return (w > 0.01 and h > 0.01 and      # width and height > 1% of image size
            x - w/2 >= 0 and x + w/2 <= 1 and  # bbox inside horizontal image bounds
            y - h/2 >= 0 and y + h/2 <= 1)      # bbox inside vertical image bounds

# --- 4. Function to perform augmentation on one image and save results ---
def augment_image(image_path, label_path, save_id):
    # Read the original image using OpenCV
    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    bboxes, class_labels = [], []

    # Read bounding box labels from YOLO label file
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            cls = int(parts[0])                   # class label
            x, y, bw, bh = map(float, parts[1:]) # bbox in YOLO format

            # Only keep valid bounding boxes
            if is_valid_bbox(x, y, bw, bh):
                bboxes.append([x, y, bw, bh])
                class_labels.append(str(cls))

    # Skip images with no valid bbox
    if not bboxes:
        return

    # Try applying augmentation transforms
    try:
        transformed = transform(image=img, bboxes=bboxes, class_labels=class_labels)
    except Exception:
        # If augmentation causes errors (e.g., invalid bbox), skip this image
        return

    # If all bounding boxes were removed after augmentation, skip saving
    if not transformed['bboxes']:
        return

    # Define output paths for augmented image and label
    out_img = os.path.join(aug_img_dir, f"aug_{save_id}.jpg")
    out_lbl = os.path.join(aug_lbl_dir, f"aug_{save_id}.txt")

    # Save augmented image
    cv2.imwrite(out_img, transformed['image'])

    # Save updated label file with transformed bounding boxes
    with open(out_lbl, 'w') as f:
        for cls, box in zip(transformed['class_labels'], transformed['bboxes']):
            f.write(f"{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n")

# --- 5. Loop over all label files to augment each image multiple times ---
save_id = 0  # Counter to name augmented files uniquely

for fname in os.listdir(label_dir):
    if not fname.endswith(".txt"):  # Process only label files
        continue

    img_path = os.path.join(image_dir, fname.replace(".txt", ".jpg"))
    lbl_path = os.path.join(label_dir, fname)

    # Skip if corresponding image file does not exist
    if not os.path.exists(img_path):
        continue

    # Augment each image 3 times
    for _ in range(3):
        augment_image(img_path, lbl_path, save_id)
        save_id += 1

print(f"Created {save_id} valid augmented images.")


In [ ]:
import shutil
import os

def merge_augmented_data(aug_img_dir, aug_lbl_dir, main_img_dir, main_lbl_dir):
    """
    Copy augmented images and labels into the main training dataset directories.

    Args:
        aug_img_dir (str): directory containing augmented images
        aug_lbl_dir (str): directory containing augmented label files
        main_img_dir (str): main training images directory to merge into
        main_lbl_dir (str): main training labels directory to merge into
    """
    img_count = 0

    # Copy all augmented images to main images folder
    for f in os.listdir(aug_img_dir):
        src_path = os.path.join(aug_img_dir, f)
        dst_path = os.path.join(main_img_dir, f)
        shutil.copy(src_path, dst_path)
        img_count += 1

    # Copy all augmented labels to main labels folder
    for f in os.listdir(aug_lbl_dir):
        src_path = os.path.join(aug_lbl_dir, f)
        dst_path = os.path.join(main_lbl_dir, f)
        shutil.copy(src_path, dst_path)

    print(f"Merged {img_count} augmented images into the training dataset.")

# Call the function to merge augmented data into main dataset
merge_augmented_data(aug_img_dir, aug_lbl_dir, image_dir, label_dir)


In [ ]:
import os
import cv2
import numpy as np
import random
import shutil

# === Base paths for original dataset ===
base_dir = "/kaggle/working/animal_dataset/train"
image_dir = os.path.join(base_dir, "images")   # Original images folder
label_dir = os.path.join(base_dir, "labels")   # Original labels folder

# Output directories for augmented data
mosaic_out = os.path.join(base_dir, "mosaic")
cutmix_out = os.path.join(base_dir, "cutmix")

# Create folders to save augmented images and labels (if they don't exist)
os.makedirs(mosaic_out + "/images", exist_ok=True)
os.makedirs(mosaic_out + "/labels", exist_ok=True)
os.makedirs(cutmix_out + "/images", exist_ok=True)
os.makedirs(cutmix_out + "/labels", exist_ok=True)


# --- Function to load YOLO labels and convert to absolute bbox coordinates ---
def load_yolo_label(label_path, img_w, img_h):
    boxes, classes = [], []
    with open(label_path, 'r') as f:
        for line in f:
            cls, x, y, w, h = map(float, line.strip().split())
            if w <= 0 or h <= 0:
                continue  # skip invalid boxes
            # Convert normalized YOLO bbox to absolute coordinates
            x1 = (x - w / 2) * img_w
            y1 = (y - h / 2) * img_h
            x2 = (x + w / 2) * img_w
            y2 = (y + h / 2) * img_h
            boxes.append([x1, y1, x2, y2])
            classes.append(int(cls))
    return boxes, classes


# --- Function to save bounding boxes back to YOLO format (normalized) ---
def save_yolo_labels(path, boxes, classes, img_w, img_h):
    with open(path, 'w') as f:
        for box, cls in zip(boxes, classes):
            x1, y1, x2, y2 = box
            # Convert absolute bbox to YOLO format (normalized center x,y and width, height)
            x = ((x1 + x2) / 2) / img_w
            y = ((y1 + y2) / 2) / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h
            # Filter out invalid or out-of-bounds boxes
            if w <= 0 or h <= 0 or x < 0 or x > 1 or y < 0 or y > 1:
                continue
            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")


# --- Mosaic augmentation ---
def mosaic_augment(n_samples=10, out_size=640):
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
    for i in range(n_samples):
        # Randomly select 4 images to combine in mosaic
        selected = random.sample(image_files, 4)
        # Create empty large canvas for mosaic image (2x output size)
        mosaic_img = np.full((out_size * 2, out_size * 2, 3), 114, dtype=np.uint8)
        labels, classes = [], []

        for idx, fname in enumerate(selected):
            img_path = os.path.join(image_dir, fname)
            lbl_path = os.path.join(label_dir, fname.replace(".jpg", ".txt"))
            img = cv2.imread(img_path)
            h, w = img.shape[:2]
            boxes, clss = load_yolo_label(lbl_path, w, h)
            # Resize image to out_size x out_size
            img = cv2.resize(img, (out_size, out_size))
            # Calculate offset in mosaic canvas
            x_off = (idx % 2) * out_size
            y_off = (idx // 2) * out_size
            # Place image in mosaic
            mosaic_img[y_off:y_off+out_size, x_off:x_off+out_size] = img

            # Adjust bounding boxes with new offsets and scale
            for box, cls in zip(boxes, clss):
                x1, y1, x2, y2 = box
                x1 = x1 * out_size / w + x_off
                y1 = y1 * out_size / h + y_off
                x2 = x2 * out_size / w + x_off
                y2 = y2 * out_size / h + y_off
                labels.append([x1, y1, x2, y2])
                classes.append(cls)

        # Save mosaic image and labels
        out_img = os.path.join(mosaic_out, "images", f"mosaic_{i}.jpg")
        out_lbl = os.path.join(mosaic_out, "labels", f"mosaic_{i}.txt")
        cv2.imwrite(out_img, mosaic_img)
        save_yolo_labels(out_lbl, labels, classes, out_size * 2, out_size * 2)


# --- CutMix augmentation ---
def cutmix_augment(n_samples=10, out_size=640):
    image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]
    for i in range(n_samples):
        # Randomly select 2 images
        a, b = random.sample(image_files, 2)
        img_a = cv2.imread(os.path.join(image_dir, a))
        img_b = cv2.imread(os.path.join(image_dir, b))
        # Resize both images to out_size x out_size
        img_a = cv2.resize(img_a, (out_size, out_size))
        img_b = cv2.resize(img_b, (out_size, out_size))

        # Randomly choose a cut point in the image
        cut_x = random.randint(out_size//4, 3*out_size//4)
        cut_y = random.randint(out_size//4, 3*out_size//4)
        # Replace bottom-right part of img_a with that part of img_b
        img_a[cut_y:, cut_x:] = img_b[cut_y:, cut_x:]

        labels, classes = [], []
        # Combine labels from both images, adjusting for cut region
        for fname, offset_x, offset_y in [(a, 0, 0), (b, cut_x, cut_y)]:
            lbl_path = os.path.join(label_dir, fname.replace(".jpg", ".txt"))
            boxes, clss = load_yolo_label(lbl_path, out_size, out_size)
            for box, cls in zip(boxes, clss):
                x1, y1, x2, y2 = box
                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2
                # Filter out boxes from img_b that fall outside the cut area
                if fname == b and (cx < cut_x or cy < cut_y):
                    continue
                labels.append([x1, y1, x2, y2])
                classes.append(cls)

        # Save augmented image and labels
        out_img = os.path.join(cutmix_out, "images", f"cutmix_{i}.jpg")
        out_lbl = os.path.join(cutmix_out, "labels", f"cutmix_{i}.txt")
        cv2.imwrite(out_img, img_a)
        save_yolo_labels(out_lbl, labels, classes, out_size, out_size)


# --- Function to merge augmented data back into main dataset folders ---
def merge_augmented(aug_dir):
    # Copy images
    for f in os.listdir(os.path.join(aug_dir, "images")):
        shutil.copy(os.path.join(aug_dir, "images", f), os.path.join(image_dir, f))
    # Copy labels
    for f in os.listdir(os.path.join(aug_dir, "labels")):
        shutil.copy(os.path.join(aug_dir, "labels", f), os.path.join(label_dir, f))

# === Run augmentations ===
mosaic_augment(n_samples=20)   # Generate 20 mosaic augmented images
cutmix_augment(n_samples=20)   # Generate 20 cutmix augmented images

# === Merge augmented data back to main dataset folders ===
merge_augmented(mosaic_out)
merge_augmented(cutmix_out)

print("Mosaic and CutMix augmentations completed and merged successfully.")


In [ ]:
from ultralytics import YOLO

# Load the pre-trained YOLOv8 nano model weights
model = YOLO("yolov8n.pt")

# Start training with your dataset and specified parameters
model.train(
    data="/kaggle/working/animal_dataset/data.yaml",  # Path to your dataset YAML config
    epochs=5,              # Number of training epochs
    imgsz=640,             # Input image size (resize images to 640x640)
    batch=16,              # Batch size for training
    augment=True,          # Enable built-in augmentation during training
    workers=2,             # Number of data loader workers
    name="animal_aug_train", # Name for the training run (used for saving weights and logs)
    amp=True               # Enable Automatic Mixed Precision for faster training with less memory
)


In [ ]:
from ultralytics import YOLO

# Load the trained YOLOv8 model weights from the best checkpoint
model = YOLO("/kaggle/working/runs/detect/animal_aug_train/weights/best.pt")

# Run prediction on all images in the test images folder
model.predict(
    source="/kaggle/working/animal_dataset/test/images",  # Folder containing test images
    conf=0.25,           # Confidence threshold to filter detections
    save=True,           # Save images with drawn bounding boxes to disk
    save_txt=True,       # Save detection results as YOLO-format text files
    save_conf=True,      # Include confidence scores in saved text files
    project="runs/detect",   # Base output directory
    name="animal_test_pred", # Folder name for this prediction run
    exist_ok=True           # Overwrite existing output folder if it exists
)


In [ ]:
# Run validation (evaluation) on the test split using your dataset YAML
metrics = model.val(
    data="/kaggle/working/animal_dataset/data.yaml",  # Dataset config file path
    split="test",   # Use the test split defined in your YAML
    imgsz=640       # Input image size for evaluation
)

# Access metrics related to bounding boxes from the result object
precision = metrics.box.p.mean()    # Mean precision across classes
recall = metrics.box.r.mean()       # Mean recall across classes
map50 = metrics.box.map50            # mAP at IoU 0.5
map_5095 = metrics.box.map           # mAP averaged over IoU thresholds 0.5 to 0.95

# Calculate F1-score from precision and recall with epsilon to avoid division by zero
f1_score = 2 * (precision * recall) / (precision + recall + 1e-6)

# Print out the evaluation results
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1_score:.4f}")
print(f"mAP@0.5: {map50:.4f}")
print(f"mAP@0.5:0.95: {map_5095:.4f}")


In [ ]:

results = model.predict(source="/kaggle/working/animal_dataset/test/images", save=False, conf=0.25)


In [ ]:
for i, result in enumerate(results):
    print(f"\n Image {i+1}:")
    boxes = result.boxes  # lấy bounding boxes
    
    for j, box in enumerate(boxes):
        xyxy = box.xyxy[0].cpu().numpy()     # [x1, y1, x2, y2]
        conf = box.conf[0].item()            # Confidence
        cls = int(box.cls[0].item())         # Class ID
        label = model.names[cls]             # Class name từ model.names

        print(f"   Box {j+1}: {label} | Conf: {conf:.2f} | Box: {xyxy}")


In [ ]:
# Calculate the total number of predicted bounding boxes across all results
total_boxes = sum(len(r.boxes) for r in results)

print(f"\nTotal number of bounding boxes predicted: {total_boxes}")


In [ ]:
for i, name in enumerate(model.names.values()):
    p, r, ap50, ap = metrics.class_result(i)
    print(f"Class: {name}")
    print(f"  Precision: {p:.4f}")
    print(f"  Recall: {r:.4f}")
    print(f"  AP@0.5: {ap50:.4f}")
    print(f"  AP@0.5:0.95: {ap:.4f}")


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque
import pandas as pd

# ======== Helper Classes ========
class SpeciesRiskConfig:
    def __init__(self, risk_dict=None):
        # Dictionary mapping species names to their risk scores
        self.risk_dict = risk_dict or {'kangaroo': 90, 'koala': 10}
    def score(self, name: str) -> int:
        # Return risk score for species name (case-insensitive), default 0
        return self.risk_dict.get(name.lower(), 0)

class RoadAreaChecker:
    def __init__(self, polygon_points, img_w, img_h):
        # Create a boolean mask for polygon representing the road area
        self.mask = self._poly_to_mask(polygon_points, img_w, img_h)
    def _poly_to_mask(self, pts, w, h):
        # Convert polygon points to a mask of shape (h, w)
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [np.array(pts, dtype=np.int32)], 1)
        return mask.astype(bool)
    def is_near_road(self, bbox):
        # Check if bounding box overlaps with the road mask
        x1, y1, x2, y2 = map(int, bbox)
        x1, y1 = max(x1, 0), max(y1, 0)
        x2, y2 = min(x2, self.mask.shape[1]-1), min(y2, self.mask.shape[0]-1)
        roi = self.mask[y1:y2+1, x1:x2+1]
        return roi.any()

class DistanceEstimator:
    def __init__(self, img_h, near_ratio=0.4):
        # Initialize with image height and ratio threshold for "close"
        self.img_h = img_h
        self.near_ratio = near_ratio
    def is_close(self, bbox):
        # Determine if bounding box is close based on its height ratio to image height
        x1, y1, x2, y2 = bbox
        box_h = y2 - y1
        return (box_h / self.img_h) >= self.near_ratio

class DayNightDetector:
    def __init__(self, threshold=60):
        # Threshold to distinguish day/night based on luminance
        self.threshold = threshold
    def get_time_of_day(self, img_bgr):
        # Convert BGR image to YCrCb and calculate mean luminance to determine day or night
        y_cr_cb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
        y_mean  = y_cr_cb[..., 0].mean()
        return 'night' if y_mean < self.threshold else 'day'

class RiskSmoother:
    def __init__(self, window=5, hi_thresh=0.6):
        # Initialize buffer for smoothing risk over a sliding window of frames
        self.window = window
        self.hi_thresh = hi_thresh
        self.buffer = deque(maxlen=window)
    def update(self, risk_lvl: str) -> str:
        # Append current risk level and determine smoothed risk level
        self.buffer.append(risk_lvl)
        if len(self.buffer) < self.window:
            return risk_lvl
        high_ratio = sum(r=='HIGH' for r in self.buffer)/self.window
        return 'HIGH' if high_ratio >= self.hi_thresh else self.buffer[-1]

# ======== Main Class: ThreatAssessmentAgentV2 ========
class ThreatAssessmentAgentV2:
    def __init__(self,
                 species_risk: SpeciesRiskConfig,
                 road_checker : RoadAreaChecker,
                 dist_est     : DistanceEstimator,
                 day_night    : DayNightDetector,
                 smoother     : RiskSmoother,
                 high_score=70,
                 med_score=40,
                 conf_thr=0.6):
        # Initialize all components and thresholds for risk scoring
        self.species_risk  = species_risk
        self.road_checker  = road_checker
        self.dist_est      = dist_est
        self.day_night_det = day_night
        self.smoother      = smoother
        self.high_score    = high_score
        self.med_score     = med_score
        self.conf_thr      = conf_thr

    def assess_risk(self, img_bgr, det):
        """
        Compute risk level based on detection info and image context.

        Args:
            img_bgr: The original BGR image frame
            det: Dictionary with detection info containing 'class_name', 'confidence', 'bbox', optional 'time_of_day'

        Returns:
            Smoothed risk level string: "LOW", "MEDIUM", or "HIGH"
        """
        cls   = det['class_name'].lower()
        conf  = det['confidence']
        bbox  = det['bbox']
        # Use provided time_of_day or compute from image brightness
        tod = det.get('time_of_day') or self.day_night_det.get_time_of_day(img_bgr)
        score = self.species_risk.score(cls)
        near_road = self.road_checker.is_near_road(bbox)
        close_obj = self.dist_est.is_close(bbox)

        risk = 'LOW'
        # High risk if species risk is high and object is near road or close
        if score >= self.high_score and (near_road or close_obj):
            risk = 'HIGH'
        # Medium risk if species risk moderate, night time, and confidence high enough
        elif (score >= self.med_score and tod == 'night' and conf >= self.conf_thr):
            risk = 'MEDIUM'
        # Return the smoothed risk level
        return self.smoother.update(risk)

# ======== Paths and YOLO Model Setup ========
model_path = "/kaggle/working/runs/detect/animal_aug_train/weights/best.pt"
image_path = "/kaggle/working/animal_dataset/test/images/00001_jpg.rf.a47ad67a15d9815ec15eaa4ebbb936ef.jpg"

img = cv2.imread(image_path)
H, W = img.shape[:2]
model = YOLO(model_path)
results = model(img)[0]

# ======== Agent Configuration ========
species_cfg = SpeciesRiskConfig({'kangaroo': 90, 'koala': 10})
road_poly   = [(0, H-130), (W, H-130), (W, H), (0, H)]
road_chk    = RoadAreaChecker(road_poly, img_w=W, img_h=H)
dist_est    = DistanceEstimator(img_h=H, near_ratio=0.45)
daynight    = DayNightDetector(threshold=70)
smoother    = RiskSmoother(window=5, hi_thresh=0.6)

agent = ThreatAssessmentAgentV2(
    species_risk=species_cfg,
    road_checker=road_chk,
    dist_est=dist_est,
    day_night=daynight,
    smoother=smoother,
    high_score=70,
    med_score=40,
    conf_thr=0.6
)

# ======== Risk Assessment from YOLO detections ========
outputs = []
for r in results.boxes:
    det = {
        'class_name' : model.names[int(r.cls)],
        'confidence' : float(r.conf),
        'bbox': r.xyxy[0].tolist(),  # Take first row and convert to list
    }
    risk = agent.assess_risk(img, det)
    outputs.append({
        'class'     : det['class_name'],
        'confidence': det['confidence'],
        'bbox'      : det['bbox'],
        'risk'      : risk
    })

# ======== Display results in a pandas DataFrame ========
df = pd.DataFrame(outputs)
display(df)


In [ ]:
class CommunicationAgent:
    def __init__(self, enable_mobile_alert=False):
        # Initialize communication agent
        # enable_mobile_alert: If True, also send alerts to mobile app
        self.enable_mobile_alert = enable_mobile_alert

    def generate_message(self, risk_level):
        # Generate alert message based on risk level
        if risk_level == "HIGH":
            return " [CRITICAL] HIGH risk detected! Immediate action required."
        elif risk_level == "MEDIUM":
            return " [WARNING] Medium risk detected. Monitor the situation."
        else:
            return " [SAFE] No significant threat detected."

    def send_alert(self, message):
        # Display alert on road display system
        print(" [ROAD DISPLAY SYSTEM]")
        print(message)
        # Optionally send alert to mobile app
        if self.enable_mobile_alert:
            print(" [MOBILE APP NOTIFICATION]")
            print(message)

    def handle_risk(self, risk_level):
        # Handle risk level by generating and sending alert if needed
        if risk_level in ["HIGH", "MEDIUM"]:
            msg = self.generate_message(risk_level)
            self.send_alert(msg)
        else:
            print(" [STATUS] LOW risk - no alert needed.")

# Example usage:
risk_level = "HIGH"  # or "MEDIUM", "LOW"
comm_agent = CommunicationAgent(enable_mobile_alert=True)
comm_agent.handle_risk(risk_level)


In [ ]:
risk_level = "HIGH"  
comm_agent = CommunicationAgent(enable_mobile_alert=True)
comm_agent.handle_risk(risk_level)


In [ ]:
import datetime

class MonitoringAgent:
    def __init__(self, log_to_file=True, log_path="monitoring_log.txt"):
        """
        Initialize MonitoringAgent.

        Args:
            log_to_file (bool): Whether to log to a file (True) or print to console (False).
            log_path (str): Path to the log file if logging to file.
        """
        self.log_to_file = log_to_file
        self.log_path = log_path

    def _write(self, text):
        """
        Internal helper to write log lines with timestamp.

        Args:
            text (str): The text message to log.
        """
        timestamp = datetime.datetime.now().strftime("[%Y-%m-%d %H:%M:%S]")
        line = f"{timestamp} {text}"

        if self.log_to_file:
            # Append the line to log file
            with open(self.log_path, "a") as f:
                f.write(line + "\n")
        else:
            # Print to console if logging to file is disabled
            print(line)

    def log_detection_event(self, detections):
        """
        Log all detection events with class, confidence, and bounding box.

        Args:
            detections (list): List of detection dicts with keys 'class', 'confidence', 'bbox'.
        """
        self._write(" Detection Event:")
        for det in detections:
            self._write(f" - {det['class']} (conf={det['confidence']:.2f}) at {det['bbox']}")

    def log_risk_assessment(self, assessed):
        """
        Log risk assessment for each detected object.

        Args:
            assessed (list): List of dicts with keys 'class' and 'risk'.
        """
        self._write(" Risk Assessment:")
        for obj in assessed:
            self._write(f" - {obj['class']} → Risk = {obj['risk']}")

    def log_alert_status(self, risk_level):
        """
        Log whether an alert has been issued based on overall risk level.

        Args:
            risk_level (str): Overall risk level (e.g., "HIGH", "MEDIUM", "LOW").
        """
        alert = "YES" if risk_level in ["HIGH", "MEDIUM"] else "NO"
        self._write(f" Alert Issued: {alert} (Overall Risk = {risk_level})")


# === Example usage ===
final_risk = "HIGH"  # Overall risk level

monitor = MonitoringAgent(log_to_file=True)  # Initialize with file logging enabled

# Log detected objects
monitor.log_detection_event(outputs)

# Log assessed risks for detected objects
monitor.log_risk_assessment(outputs)

# Log whether alert was issued based on overall risk
monitor.log_alert_status(final_risk)
